# simdash data: notebook template

A starting point for analysing what simdash records, outside the web app. It reads the per-day SQLite
databases directly (layout in [docs/data.md](../docs/data.md#databases)) into pandas.

**Needs:** `pip install pandas matplotlib zstandard` (in any environment with Jupyter; simdash itself is not
imported). It only reads: simdash can keep recording while this runs.

Sections:
1. Open the databases
2. Every session, from every day
3. Pick a track and a car: all its laps
4. One lap's samples, two laps compared (delta time)
5. Track map coloured by speed
6. Tyre temperatures along a lap
7. Corners: time per corner over every lap
8. A race: results and positions of every car
9. Change logs: settings and weather as they were

## 1. Open the databases

In [ ]:
import os, sqlite3
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path(os.environ.get("SIMDASH_DATA", Path.home() / ".local/share/simdash"))
DAYS = DATA / "days"
# compressed days are unpacked here (the notebook's own cache, separate from simdash's)
CACHE = Path(os.environ.get("SIMDASH_NB_CACHE", Path.home() / ".cache/simdash-notebook"))
pd.set_option("display.width", 200)


def list_days():
    """Days with a database, newest first: plain .sqlite, or compressed .sqlite.zst."""
    names = {p.name.split(".")[0] for p in DAYS.glob("????-??-??.sqlite")}
    names |= {p.name.split(".")[0] for p in DAYS.glob("????-??-??.sqlite.zst")}
    return sorted(names, reverse=True)


def open_day(day, samples=True):
    """Connection to one day. samples=False reads a compressed day's small summary (sessions and laps only)
    instead of unpacking it."""
    plain = DAYS / f"{day}.sqlite"
    if plain.exists():
        path = plain
    elif not samples and (DAYS / f"{day}.summary.sqlite").exists():
        path = DAYS / f"{day}.summary.sqlite"
    else:
        import zstandard
        src, path = DAYS / f"{day}.sqlite.zst", CACHE / f"{day}.sqlite"
        if not path.exists() or path.stat().st_mtime < src.stat().st_mtime:
            CACHE.mkdir(parents=True, exist_ok=True)
            with open(src, "rb") as f, open(path.with_suffix(".tmp"), "wb") as g:
                zstandard.ZstdDecompressor().copy_stream(f, g)
            path.with_suffix(".tmp").replace(path)
    con = sqlite3.connect(path)
    # v_samples uses atan2/degrees: provide them in case SQLite was built without math functions
    con.create_function("atan2", 2, np.arctan2, deterministic=True)
    con.create_function("degrees", 1, np.degrees, deterministic=True)
    return con


def local_time(ts):
    """Unix seconds (as recorded) to local date-times."""
    return pd.to_datetime(ts, unit="s", utc=True).dt.tz_convert(datetime.now().astimezone().tzinfo).dt.tz_localize(None)


def query(day, sql, params=(), samples=True):
    with open_day(day, samples) as con:
        return pd.read_sql_query(sql, con, params=params)


print(DATA, "|", len(list_days()), "days:", ", ".join(list_days()))

## 2. Every session, from every day

Session ids are unique only inside their day, so a session is named `day:id` (as in simdash's
`/#session=2026-11-06:13` links). Tracks and cars are matched across days by name.

In [ ]:
sessions = pd.concat([query(d, "SELECT * FROM v_sessions", samples=False).assign(day=d) for d in list_days()],
                     ignore_index=True)
sessions["ref"] = sessions["day"] + ":" + sessions["id"].astype(str)
sessions["started"] = local_time(sessions["started_at"])
sessions[["ref", "started", "session_type", "track", "variation", "car", "laps", "best_lap", "driving_s"]] \
    .sort_values("started", ascending=False).head(15)

In [ ]:
# driving time and laps per track and car
(sessions.groupby(["track", "car"])
 .agg(sessions=("ref", "count"), laps=("laps", "sum"), hours=("driving_s", lambda s: round(s.sum() / 3600, 2)),
      best=("best_lap", "min"))
 .sort_values("hours", ascending=False))

## 3. Pick a track and a car: all its laps

In [ ]:
# change these; by default the track and car you drove the most
top = sessions.groupby(["track", "variation", "car"])["driving_s"].sum().idxmax()
TRACK, VARIATION, CAR = top
print(TRACK, "/", VARIATION, "/", CAR)

In [ ]:
combo = sessions[(sessions.track == TRACK) & (sessions.variation == VARIATION) & (sessions.car == CAR)]
laps = pd.concat([
    query(s.day, "SELECT * FROM lap WHERE session_id = ? AND lap_time > 0", (int(s.id),), samples=False)
    .assign(ref=s.ref, day=s.day, session_type=s.session_type, started=s.started)
    for s in combo.itertuples()], ignore_index=True)
laps["valid"] = laps["valid"].astype(bool)
best = laps[laps.valid].sort_values("lap_time").iloc[0]
print(f"{len(laps)} laps, {laps.valid.sum()} valid; best {best.lap_time:.3f} s ({best.ref} lap {best.lap})")
laps[["ref", "session_type", "lap", "lap_time", "valid", "s1", "s2", "s3", "top_speed", "fuel_used"]] \
    .sort_values("lap_time").head(10)

In [ ]:
# lap times over time, valid laps within 107 % of the best
ok = laps[laps.valid & (laps.lap_time < best.lap_time * 1.07)]
fig, ax = plt.subplots(figsize=(10, 3.5))
for t, g in ok.groupby("session_type"):
    ax.scatter(g.started + pd.to_timedelta(g.lap * 2, unit="min"), g.lap_time, s=14, label=t)
ax.set_ylabel("lap time (s)"); ax.set_title(f"{TRACK} · {CAR}"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 4. One lap's samples, two laps compared

`v_samples` gives your car's samples in the app's units (km/h, G, degrees). Lap N is the rows with `lap = N`.
Laps on old, slimmed days keep only every 10th sample (2 Hz), except the good ones; `len(lap)` tells.

In [ ]:
def load_lap(ref, lap):
    day, sid = ref.split(":")
    return query(day, "SELECT * FROM v_samples WHERE session_id = ? AND lap = ? AND lap_time >= 0 ORDER BY ts",
                 (int(sid), int(lap)))


def time_at(L, dist):
    """Lap time at given lap distances, interpolated."""
    return np.interp(dist, L.lap_dist, L.lap_time)


A = load_lap(best.ref, best.lap)
# the second best valid lap, from any session
other = laps[laps.valid].sort_values("lap_time").iloc[1]
B = load_lap(other.ref, other.lap)
print(len(A), "and", len(B), "samples")
A.head()

In [ ]:
grid = np.linspace(0, min(A.lap_dist.max(), B.lap_dist.max()), 2000)
delta = time_at(B, grid) - time_at(A, grid)   # > 0: the other lap is behind at that point

fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True, gridspec_kw={"height_ratios": [1, 2, 1, 1]})
ax[0].plot(grid, delta, color="tab:orange"); ax[0].axhline(0, color="grey", lw=0.5); ax[0].set_ylabel("Δ s")
for L, name, style in ((A, f"best {best.lap_time:.3f}", "-"), (B, f"other {other.lap_time:.3f}", ":")):
    ax[1].plot(L.lap_dist, L.speed, style, label=name)
    ax[2].plot(L.lap_dist, L.throttle * 100, style, color="tab:green")
    ax[2].plot(L.lap_dist, L.brake * 100, style, color="tab:red")
    ax[3].plot(L.lap_dist, L.steer * 100, style, color="tab:grey")
ax[1].set_ylabel("km/h"); ax[1].legend(); ax[2].set_ylabel("pedals %"); ax[3].set_ylabel("steering %")
ax[3].set_xlabel("lap distance (m)")
for a in ax: a.grid(alpha=0.3)
plt.show()

## 5. Track map coloured by speed

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
sc = ax.scatter(-A.x, A.z, c=A.speed, s=3, cmap="viridis")   # simdash draws -x, z; mirror or rotate to taste
ax.set_aspect("equal"); ax.axis("off"); fig.colorbar(sc, ax=ax, shrink=0.6, label="km/h")
ax.set_title(f"{TRACK}, best lap")
plt.show()

## 6. Tyre temperatures along a lap

Per-wheel data is in the `wheel` table, four rows per tick (`wheel` 0 FL, 1 FR, 2 RL, 3 RR), joined to the
lap through `tick` and `telemetry`.

In [ ]:
def load_wheels(ref, lap, cols=("tyre_temp_c", "brake_temp_c", "wear")):
    day, sid = ref.split(":")
    df = query(day, f"""SELECT t.ts, e.lap_dist, w.wheel, {", ".join("w." + c for c in cols)}
                        FROM tick t JOIN telemetry e ON e.tick_id = t.id JOIN wheel w ON w.tick_id = t.id
                        WHERE t.session_id = ? AND e.lap = ? ORDER BY t.ts""", (int(sid), int(lap)))
    df["wheel"] = df.wheel.map(dict(enumerate(["FL", "FR", "RL", "RR"])))
    return df


W = load_wheels(best.ref, best.lap)
temps = W.pivot_table(index="lap_dist", columns="wheel", values="tyre_temp_c")
ax = temps.plot(figsize=(12, 3.5), title="tyre temperature (°C), best lap"); ax.grid(alpha=0.3)
plt.show()
W.groupby("wheel")[["tyre_temp_c", "brake_temp_c"]].agg(["min", "mean", "max"]).round(1)

## 7. Corners: time per corner over every lap

The same rule as simdash ([docs/corners.md](../docs/corners.md)): a corner is a speed dip of more than
10 km/h; its segment runs from the speed peak before to the speed peak after. Corners come from the best
lap, then every lap is timed over the same distances.

In [ ]:
def find_corners(L, dip=10):
    v, d = L.speed.to_numpy(), L.lap_dist.to_numpy()
    peaks, apexes, hi, lo, falling = [], [], 0, 0, False
    for i in range(1, len(v)):
        if not falling:
            if v[i] >= v[hi]: hi = i
            elif v[hi] - v[i] > dip: peaks.append(hi); falling = True; lo = i
        elif v[i] <= v[lo]: lo = i
        elif v[i] - v[lo] > dip: apexes.append(lo); falling = False; hi = i
    if not falling: peaks.append(hi)
    return pd.DataFrame([{"corner": f"C{k + 1}", "start": d[peaks[k]] if k else 0.0, "apex": d[a],
                          "end": d[peaks[k + 1]] if k < len(apexes) - 1 else d[-1]} for k, a in enumerate(apexes)])


corners = find_corners(A)
corners.round(0)

In [ ]:
# every valid lap of the combination (one query per lap: give it a moment on a long history)
rows = []
for l in laps[laps.valid].itertuples():
    L = load_lap(l.ref, l.lap)
    if len(L) < 50:
        continue
    t = time_at(L, np.r_[corners.start, corners.end.iloc[-1]])
    rows.append({"ref": l.ref, "lap": l.lap, "lap_time": l.lap_time, "samples": len(L),
                 **dict(zip(corners.corner, np.diff(t)))})
per_corner = pd.DataFrame(rows)
print(len(per_corner), "laps")
# best, median and the gap between them per corner: where consistency costs the most
summary = per_corner[corners.corner].agg(["min", "median"]).T
summary["median - best"] = summary["median"] - summary["min"]
summary.round(3).sort_values("median - best", ascending=False)

In [ ]:
ax = per_corner[corners.corner].sub(per_corner[corners.corner].min()).plot.box(figsize=(12, 3.5), showfliers=False)
ax.set_ylabel("s slower than the best"); ax.set_title("time lost per corner, every valid lap"); ax.grid(alpha=0.3)
plt.show()

## 8. A race: results and positions of every car

Other cars live in `car_motion` (positions at 2 Hz) and `car_status` (on change), keyed by `tick_id` and
`idx` (the car's slot, as in `participant`). They have no `session_id`: they belong to a session through
its tick range.

In [ ]:
# the race with the most laps (change to any ref, e.g. sessions[sessions.ref == "2026-11-06:13"].iloc[0])
races = sessions[sessions.session_type == "Race"].sort_values(["laps", "started"], ascending=False)
race = races.iloc[0] if len(races) else sessions.iloc[0]
print(race.ref, race.session_type, race.track, race.car)
query(race.day, "SELECT pos, name, car, laps_completed, best_lap, race_state, is_player FROM v_results "
      "WHERE session_id = ? ORDER BY pos IS NULL, pos", (int(race.id),)).head(10)

In [ ]:
pos = query(race.day, """
    SELECT t.ts, m.idx, m.race_pos, p.is_player, d.name
    FROM car_motion m JOIN tick t ON t.id = m.tick_id
    JOIN participant p ON p.session_id = t.session_id AND p.idx = m.idx JOIN driver d ON d.id = p.driver_id
    WHERE t.session_id = ? AND m.race_pos > 0 ORDER BY t.ts""", (int(race.id),))
pos["min"] = (pos.ts - pos.ts.min()) / 60
fig, ax = plt.subplots(figsize=(12, 5))
for (idx, me), g in pos.groupby(["idx", "is_player"]):
    ax.plot(g["min"], g.race_pos, lw=2.5 if me else 0.7, color="tab:blue" if me else "lightgrey", zorder=3 if me else 1)
ax.invert_yaxis(); ax.set_xlabel("minutes"); ax.set_ylabel("position"); ax.set_title(f"{race.track}, {race.ref}")
ax.grid(alpha=0.3)
plt.show()

## 9. Change logs: settings and weather as they were

`player_state`, `player_timing`, `event_state` and `weather` write a row only when a value changes. A row
holds until the next one of the same session: the value at a moment is the last row at or before it.

In [ ]:
day, sid = best.ref.split(":")
changes = query(day, """SELECT t.ts, s.brake_bias, s.abs_setting, s.tc_setting, s.compound_fl
                        FROM player_state s JOIN tick t ON t.id = s.tick_id WHERE t.session_id = ? ORDER BY t.ts""",
                (int(sid),))
changes["at"] = local_time(changes.ts)
# keep only the rows where one of these settings changed (the log also has rows for other player_state columns)
cols = ["brake_bias", "abs_setting", "tc_setting", "compound_fl"]
changes[(changes[cols] != changes[cols].shift()).any(axis=1)].head(10)

In [ ]:
weather = query(day, """SELECT t.ts, w.air_temp_c, w.track_temp_c, w.rain FROM weather w JOIN tick t ON t.id = w.tick_id
                        WHERE t.session_id = ? ORDER BY t.ts""", (int(sid),))
# the weather in force at every sample of the best lap: an "as of" join
pd.merge_asof(A[["ts", "lap_dist", "speed"]], weather, on="ts").iloc[::200]

## Next ideas

- Braking points and minimum speeds per corner over time (`cornerStats` in `simdash/static/app.js` is the
  reference implementation).
- Tyre temperature at corner entry against corner time, per car.
- Fuel used per lap against lap time; tyre wear over a race.
- Compare cars on the same track: the `corners` table with `per_corner` grouped by car.